# 🌐 06. Xây dựng Mạng lưới Đồng xuất hiện (Co-occurrence Network) & Centrality

**Thành viên thực hiện:** Thành viên 2 (Graph Mining Lead)

Mục tiêu:
- Xây dựng đồ thị vô hướng trọng số từ ma trận đồng xuất hiện giữa các Tướng.
- Đỉnh (Node) = Tướng, Cạnh (Edge) = Xuất hiện cùng nhau trong đội hình, Trọng số (Weight) = Tần suất / Tỷ lệ thắng.
- Phân tích cấu trúc mạng: Degree distribution, Density, Clustering coefficient.
- Trực quan hóa mạng lưới liên kết tướng.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from src.graph_miner import GraphMiner
import warnings
warnings.filterwarnings('ignore')

## 1. Nạp ma trận đồng xuất hiện

In [ ]:
cooc_freq = pd.read_pickle('../data/processed/cooc_frequency.pkl')
cooc_wr = pd.read_pickle('../data/processed/cooc_winrate.pkl')
print(f'Kích thước ma trận: {cooc_freq.shape}')

## 2. Khởi tạo Đồ thị bằng GraphMiner

In [ ]:
gm = GraphMiner()
G = gm.build_network(cooc_wr, min_weight=0.15)
stats = gm.get_network_stats(G)
print('Thống kê mạng lưới:')
for k, v in stats.items():
    print(f'  {k}: {v}')

## 3. Phân tích Phân phối Bậc (Degree Distribution)

In [ ]:
degrees = [d for n, d in G.degree()]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(degrees, kde=True, color='purple', ax=axes[0])
axes[0].set_title('Phân phối Bậc (Degree Distribution)')
axes[0].set_xlabel('Degree')

axes[1].scatter(range(len(degrees)), sorted(degrees, reverse=True), color='red')
axes[1].set_title('Degree Rank Plot')
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Degree')
plt.tight_layout()
plt.show()

## 4. Phân tích Centrality Metrics

In [ ]:
centrality_df = gm.compute_centrality_metrics(G)
display(centrality_df.head(15))

# Lưu đồ thị và metrics cho các notebook sau
nx.write_gpickle(G, '../data/processed/network_graph.gpickle') if hasattr(nx, 'write_gpickle') else None
centrality_df.to_pickle('../data/processed/centrality_df.pkl')

## 5. Trực quan hóa Mạng lưới Tướng (Spring Layout)

In [ ]:
fig = gm.visualize_network(G, figsize=(12, 10), output_path='../reports/network_topology.png')
plt.show()

## 6. Nhận xét
- Đồ thị có cấu trúc cụm rõ rệt, kết nối dày đặc giữa các tướng có chung Tộc/Hệ.
- Các tướng linh hoạt (Flex picks) đóng vai trò là cầu nối (Bridge) giữa các nhóm đội hình khác nhau.